### 1. Problem, data and KPIs definition, and Assumptions

#### Defining Material classes (Slab and Plate) and orders

The data model explicitly captures the key operational compatibility requirements specified in the problem:

- **Steel grade:** Both slabs and plates carry a grade attribute. An allocation is only considered feasible when the slab grade is compatible with the plate requirement.
- **Quality specification:** Slabs and plates also contain a quality specification, ensuring that material is only allocated where the required quality is satisfied.
- **Customer-specific requirements:** Customer restrictions are associated with the order and can impose additional constraints, such as approved slab suppliers or required surface classifications.
- **Separation of concerns:** These attributes are stored in the relevant domain objects, while a separate compatibility function determines whether a particular slab–plate allocation is feasible.

In [ ]:
class Slab:

    def __init__(
        self,
        slab_id,
        length,
        grade,
        quality,
        supplier,
        surface_class,
        status="available"
    ):
        self.slab_id = slab_id
        self.length = length
        self.grade = grade
        self.quality = quality
        self.supplier = supplier
        self.surface_class = surface_class
        self.status = status


class Plate:

    def __init__(
        self,
        plate_id,
        order_id,
        length,
        width,
        thickness,
        grade,
        quality
    ):
        self.plate_id = plate_id
        self.order_id = order_id
        self.length = length
        self.width = width
        self.thickness = thickness
        self.grade = grade
        self.quality = quality

class Order:

    def __init__(
        self,
        order_id,
        customer,
        plates,
        allowed_suppliers=None,
        required_surface_class=None,
        require_full_fulfilment=True
    ):
        self.order_id = order_id
        self.customer = customer
        self.plates = plates
        self.allowed_suppliers = allowed_suppliers
        self.required_surface_class = required_surface_class
        self.require_full_fulfilment = require_full_fulfilment

#### Defining compatibility and crucial legth calculating function

For demonstration purposes, for two different grades, separate rolling/yield allowances are assumed. In real-world, this function would be replaced by the plant's established process model.

In [ ]:
def is_compatible(slab, plate, order):

    # Steel grade
    if slab.grade != plate.grade:
        return False

    # Quality specification
    if slab.quality != plate.quality:
        return False

    # Customer-specific supplier restriction
    if order.allowed_suppliers is not None:
        if slab.supplier not in order.allowed_suppliers:
            return False

    # Customer-specific surface restriction
    if order.required_surface_class is not None:
        if slab.surface_class != order.required_surface_class:
            return False

    # Inventory status
    if slab.status != "available":
        return False

    return True


def calculate_required_length(slab, plate):
    
    if slab.grade == "G1":
        yield_factor = 1.06

    elif slab.grade == "G2":
        yield_factor = 1.12

    else:
        raise ValueError(
            f"Unsupported slab grade: {slab.grade}"
        )

    return plate.length * yield_factor

### 2. Defining KPIs

In [10]:
class Allocation:

    def __init__(self, slab, plate, order):

        if not is_compatible(slab, plate, order):
            raise ValueError(
                f"Slab {slab.slab_id} is not compatible "
                f"with Plate {plate.plate_id}"
            )

        self.slab = slab
        self.plate = plate
        self.order = order

        self.required_length = calculate_required_length(
            slab,
            plate
        )



def calculate_material_consumption(allocations):
    """
    Calculate material consumption by slab and in total.
    """
    consumption_by_slab = {}

    for allocation in allocations:
        slab_id = allocation["slab_id"]
        required_length = allocation["required_length"]

        consumption_by_slab[slab_id] = (
            consumption_by_slab.get(slab_id, 0)
            + required_length
        )

    total_consumption = sum(consumption_by_slab.values())

    return consumption_by_slab, total_consumption



def calculate_waste(slabs, allocations):
    consumption_by_slab, _ = (
        calculate_material_consumption(allocations)
    )
    total_waste = 0.0

    for slab in slabs:
        used = consumption_by_slab.get(slab.slab_id, 0.0)
        total_waste += slab.length - used

    return total_waste


def calculate_yield_loss(slabs, allocations):
    consumption_by_slab, _ = calculate_material_consumption(
        allocations
    )

    yield_loss = 0.0

    for slab in slabs:
        if slab.slab_id in consumption_by_slab:
            yield_loss += (
                slab.length
                - consumption_by_slab[slab.slab_id]
            )

    return yield_loss
